In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

if not drive_root.is_dir():
    raise RuntimeError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")

print("Đã kết nối Drive.")

In [ ]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "bd92efd65cd21ed2add88defd3a110e931f5d573"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

if not (repo_dir / ".git").is_dir():
    raise RuntimeError("Lỗi: Thư mục dự án không phải Git repository.")

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository có origin khác với dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository có thay đổi chưa lưu. "
        "Cần kiểm tra trước khi chuyển commit."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit đang sử dụng:", target_commit)
print("Thư mục làm việc:", Path.cwd())

In [ ]:
%pip uninstall -y diffusers
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

data_module_path = repo_dir / "src" / "data.py"
data_module_path.write_bytes(uploaded_content)

print("File đã chọn:", file_name)
print("Đã cập nhật:", data_module_path)

In [ ]:
import json
from src.data import load_wikilingua, save_source_manifest

cache_dir = drive_root / "data" / "raw"
manifest_path = drive_root / "data" / "manifests" / "wikilingua_source.json"

dataset = load_wikilingua(cache_dir)

manifest = save_source_manifest(
    dataset,
    manifest_path,
    cache_dir,
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Đã lưu manifest:", manifest_path)

In [ ]:
files.download(str(manifest_path))

In [ ]:
import importlib
import src.data

importlib.reload(src.data)

source_rows = dataset["train"]

if len(source_rows) < 10:
    raise RuntimeError("Lỗi: Không đủ 10 bài để khảo sát.")

for row_index in range(10):
    row = source_rows[row_index]

    print("=" * 80)
    print("Vị trí bài:", row_index)
    print("URL:", row["url"])
    print("Kiểu dữ liệu article:", type(row["article"]).__name__)
    print(json.dumps(row["article"], ensure_ascii=False, indent=2))

In [ ]:
pairs = src.data.convert_wikilingua_sections(source_rows)

expected_pair_count = 0

for row in source_rows:
    section_count = len(row["article"]["document"])
    expected_pair_count = expected_pair_count + section_count

if len(pairs) != expected_pair_count:
    raise RuntimeError("Lỗi: Số cặp không khớp tổng số section nguồn.")

print(json.dumps(
    {
        "article_count": len(source_rows),
        "section_count": expected_pair_count,
        "pair_count": len(pairs),
        "pair_count_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
selected_pairs = []
selected_rows = []

for pair in pairs:
    row_index = pair["source_row_index"]

    if row_index not in selected_rows:
        selected_pairs.append(pair)
        selected_rows.append(row_index)

    if len(selected_pairs) == 5:
        break

if len(selected_pairs) != 5:
    raise RuntimeError("Lỗi: Không đủ mẫu từ 5 bài khác nhau.")

sample_number = 0

for pair in selected_pairs:
    sample_number = sample_number + 1
    row_index = pair["source_row_index"]
    section_index = pair["source_section_index"]

    source_row = source_rows[row_index]
    source_article = source_row["article"]

    if pair["url"] != source_row["url"]:
        raise RuntimeError("Lỗi: URL không khớp bài nguồn.")

    for field in ["section_name", "document", "summary"]:
        if pair[field] != source_article[field][section_index]:
            raise RuntimeError(
                f"Lỗi: Trường {field} không khớp section nguồn."
            )

    print("=" * 80)
    print("Mẫu đối chiếu:", sample_number)
    print(json.dumps(pair, ensure_ascii=False, indent=2))
    print("Đối chiếu giá trị với section nguồn: đạt.")

In [15]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file preprocessing.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

module_path = repo_dir / "src" / "preprocessing.py"
module_path.write_bytes(uploaded_content)

import src.preprocessing

importlib.reload(src.preprocessing)

print("File đã chọn:", file_name)
print("Đã cập nhật:", module_path)

Saving preprocessing.py to preprocessing.py
File đã chọn: preprocessing.py
Đã cập nhật: /content/vietnamese-summarization/src/preprocessing.py


In [16]:
import json
import unicodedata

if "pairs" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu cặp từ cell 8.")

normalized_pairs = src.preprocessing.normalize_pairs(pairs)

if len(normalized_pairs) != len(pairs):
    raise RuntimeError("Lỗi: Số cặp thay đổi sau chuẩn hóa.")

changed_counts = {
    "document": 0,
    "summary": 0,
}

for pair_index in range(len(pairs)):
    original_pair = pairs[pair_index]
    normalized_pair = normalized_pairs[pair_index]

    for field in ["document", "summary"]:
        original_value = original_pair.get(field)
        normalized_value = normalized_pair.get(field)

        if isinstance(original_value, str):
            if original_value != normalized_value:
                changed_counts[field] = changed_counts[field] + 1

            if not unicodedata.is_normalized("NFC", normalized_value):
                raise RuntimeError("Lỗi: Văn bản chưa ở dạng NFC.")

            checked_value = src.preprocessing.normalize_text(
                normalized_value
            )

            if checked_value != normalized_value:
                raise RuntimeError("Lỗi: Chuẩn hóa chưa ổn định.")

        else:
            if normalized_value != original_value:
                raise RuntimeError("Lỗi: Giá trị không phải chuỗi bị thay đổi.")

print(json.dumps(
    {
        "pair_count_before": len(pairs),
        "pair_count_after": len(normalized_pairs),
        "changed_document_count": changed_counts["document"],
        "changed_summary_count": changed_counts["summary"],
        "pair_count_check_passed": True,
        "nfc_check_passed": True,
        "idempotence_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

{
  "pair_count_before": 19581,
  "pair_count_after": 19581,
  "changed_document_count": 16226,
  "changed_summary_count": 178,
  "pair_count_check_passed": true,
  "nfc_check_passed": true,
  "idempotence_check_passed": true
}


In [17]:
displayed_count = 0

for pair_index in range(len(pairs)):
    original_pair = pairs[pair_index]
    normalized_pair = normalized_pairs[pair_index]

    for field in ["document", "summary"]:
        original_value = original_pair.get(field)
        normalized_value = normalized_pair.get(field)

        if isinstance(original_value, str):
            if original_value != normalized_value:
                displayed_count = displayed_count + 1

                print("=" * 80)
                print("Mẫu đối chiếu:", displayed_count)
                print("Vị trí cặp:", pair_index)
                print("URL:", original_pair["url"])
                print("Section:", original_pair["section_name"])
                print("Trường:", field)
                print("Trước:", repr(original_value))
                print("Sau:", repr(normalized_value))

        if displayed_count == 3:
            break

    if displayed_count == 3:
        break

if displayed_count == 0:
    print("Không có trường dạng chuỗi nào thay đổi sau chuẩn hóa.")

Mẫu đối chiếu: 1
Vị trí cặp: 0
URL: https://www.wikihow.vn/Mua-%C4%91%C6%B0%E1%BB%A3c-th%E1%BB%B1c-ph%E1%BA%A9m-l%C3%A0nh-m%E1%BA%A1nh
Section: Tìm mua các sản phẩm tốt mà rẻ
Trường: document
Trước: 'Hoa quả và rau thường rẻ hơn khi vào mùa. Thường thì các cửa hàng thực phẩm sẽ giảm giá cho các mặt hàng nông sản đang vào mùa vì họ biết khách hàng luôn chờ đợi những loại nào vào thời gian nào trong năm. Hơn nữa rau quả đúng mùa bao giờ cũng ngon hơn.  Vào mùa thu, bạn nên chọn các sản phẩm như bí ngô, bí mùa đông, cải bắp và táo. Mùa đông đến, bạn hãy tìm bí mùa đông, cải bắp và các loại củ khác. Mùa xuân là mùa của các loại rau củ như củ cải, rau lá xanh, hành lá và măng tây. Mùa hè là dịp tốt để thưởng thức nhiều loại rau quả, đơn cử như dưa hấu, ngô và các loại quả mọng. Có lẽ bạn cũng nhận thấy là giá cả của rau củ quả thường rẻ nhất trong mùa hè. Ngoài ra, vì nông sản mùa hè rẻ hơn nên bạn có thể mua nhiều để đông lạnh hoặc tự đóng hộp. Có lẽ bạn thích hoa quả và rau củ tươi hơn, n